In [4]:
import os
from dotenv import load_dotenv

# Load environment variables from a .env file
load_dotenv()
 

True

In [6]:
print(os.getenv("OPENAI_API_KEY"))

In [7]:
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

# Langsmith trackign and tracing
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")
os.environ["LANGCHAIN_TRACING_V2"] = "true"

In [8]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="o1-mini")
print(llm)

client=<openai.resources.chat.completions.completions.Completions object at 0x000002276D8E0250> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x000002276E0C9D10> root_client=<openai.OpenAI object at 0x000002276D8DBCD0> root_async_client=<openai.AsyncOpenAI object at 0x000002276E0C9810> model_name='o1-mini' temperature=1.0 model_kwargs={} openai_api_key=SecretStr('')


In [9]:
result = llm.invoke("What is agentic AI?")
print(result.content)

APIConnectionError: Connection error.

In [11]:
from langchain_groq import ChatGroq
groq_llm = ChatGroq(model="qwen-qwq-32b")
result = groq_llm.invoke("Hi my name is Deepak, what is your name?")
print(result.content)


<think>
Okay, the user introduced himself as Deepak and asked for my name. I need to respond politely. Let me start by greeting him back. I should mention my name, which is Qwen. It's important to be friendly and open. Maybe I can ask him how I can assist him today. That way, it invites him to share more about what he needs help with. Let me make sure the tone is warm and not too formal. Alright, that should cover the essentials without being too verbose.
</think>

Hello Deepak! My name is Qwen. I'm a large language model developed by Alibaba Cloud. How can I assist you today?


In [12]:
### Prompt Engineering
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are an AI Enginneering expert. Provide me answer based on the question."),
        ("user", "{input}"),
    ]
)
prompt


ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an AI Enginneering expert. Provide me answer based on the question.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [13]:
from langchain_groq import ChatGroq
model = ChatGroq(model="gemma2-9b-it")
model

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000227742BFDD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000227742D8AD0>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [14]:
### chaning the model
chain =prompt| model
chain

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an AI Enginneering expert. Provide me answer based on the question.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000227742BFDD0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000227742D8AD0>, model_name='gemma2-9b-it', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [15]:
response = chain.invoke({"input": "Can you tell me something about langsmith"})
print(response.content)

You're in luck! I know quite a bit about LangSmith. 

**LangSmith** is an open-source platform developed by the **AI21 Labs** team. It's designed to simplify the process of building and fine-tuning large language models (LLMs). 

Here are some key things to know about LangSmith:

* **User-Friendly Interface:** LangSmith provides a web-based interface that makes it accessible to a wider range of users, even those without extensive coding experience.

* **Modular Architecture:** It follows a modular design, allowing users to easily integrate different components like datasets, training algorithms, and evaluation metrics.
* **Fine-Tuning Capabilities:** A central feature is its ability to fine-tune pre-trained LLMs on specific tasks or domains. This allows users to adapt powerful models to their unique needs.
* **Community Driven:** Being open-source, LangSmith benefits from a vibrant community of developers and researchers who contribute to its development and share their expertise.

**H

In [16]:
from langchain_core.output_parsers import BaseOutputParser

# Define a custom output parser
class CustomStrOutputParser(BaseOutputParser):
    def get_format_instructions(self) -> str:
        return "Please provide a plain text response."  

# Use the custom output parser
output_parser = CustomStrOutputParser()

# Ensure `prompt` and `model` are defined earlier in the code
chain = prompt | model | output_parser
response = chain.invoke({"input": "Can you tell me something about langsmith"})
print(response)

TypeError: Can't instantiate abstract class CustomStrOutputParser with abstract method parse

In [17]:
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate

output_parser = JsonOutputParser()

prompt = PromptTemplate(
    template="Answer the user query \n {format_instructions}\n {query}",
    input_variables=["query"],
    partial_variables={"format_instructions": output_parser.get_format_instructions()},
)

prompt

 

PromptTemplate(input_variables=['query'], input_types={}, partial_variables={'format_instructions': 'Return a JSON object.'}, template='Answer the user query \n {format_instructions}\n {query}')

In [18]:
chain = prompt | model | output_parser
response = chain.invoke({"query": "Can you tell me something about langsmith"})
print(response)


{'name': 'LangSmith', 'description': 'LangSmith is an open-source platform for building, sharing, and evaluating large language models (LLMs).', 'features': ['Model training and fine-tuning', 'Dataset management', 'Experiment tracking', 'Model evaluation', 'Community collaboration'], 'website': 'https://github.com/alangsmith/langsmith'}
